# Flow Matching

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.datasets import MNIST

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display, HTML
from tqdm import tqdm
from dlprog import train_progress

prog = train_progress()

plt.style.use("ggplot")
plt.rcParams["animation.embed_limit"] = 30.0 * 1024 * 1024

def show_images(imgs, ncol=5, w=128):
    _, _, h_img, w_img = imgs.shape
    h = int(h_img * w / w_img)
    imgs = transforms.Resize((h, w), antialias=True)(imgs)
    img = torchvision.utils.make_grid(imgs, nrow=ncol)
    img = transforms.functional.to_pil_image(img)
    display(img)

device = torch.accelerator.current_accelerator(check_available=True)
device

## MNIST Dataset

In [ ]:
SIZE = 28
batch_size = 64

ds = MNIST(root="./data", download=True, transform=transforms.ToTensor())
dataloader = DataLoader(ds, batch_size=batch_size, shuffle=True)
x, _ = next(iter(dataloader))
print(f"Number of images: {len(ds)}")
print(f"Image shape: {x.shape}")
show_images(x[:20])

## U-Net

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, d=128, base=10_000):
        super().__init__()
        self.d = d
        self.base = base
        idx = torch.arange(0, d, 2).float()
        self.register_buffer("idx", idx)

    def forward(self, t):
        sin = torch.sin(1000 * t.unsqueeze(-1) / self.base ** (self.idx / self.d))
        cos = torch.cos(1000 * t.unsqueeze(-1) / self.base ** (self.idx / self.d))
        e = torch.cat([sin, cos], dim=-1)
        return e


class UNet(nn.Module):
    def __init__(self, d_t=128):
        super().__init__()
        self.embed = TimeEmbedding(d=d_t)
        self.conv1 = self.get_conv_block(1, 32) # 28 -> 14
        self.conv2 = self.get_conv_block(32, 64) # 14 -> 7
        self.conv3 = self.get_conv_block(64, 128, 3, 1, 0) # 7 -> 5

        self.convT3 = self.get_convT_block(128, 64, 3, 1, 0, 0) # 5 -> 7
        self.convT2 = self.get_convT_block(64 * 2, 32) # 7 -> 14
        self.convT1 = self.get_convT_block(32 * 2, 16) # 14 -> 28

        self.out_conv = nn.Conv2d(16, 1, 1) # 28 -> 28

        self.fc_t1 = nn.Linear(d_t, 32)
        self.fc_t2 = nn.Linear(d_t, 64)
        self.fc_t3 = nn.Linear(d_t, 128)
        self.fc_t4 = nn.Linear(d_t, 128)
        self.fc_t5 = nn.Linear(d_t, 64)
        self.fc_t6 = nn.Linear(d_t, 16)

    def get_conv_block(self, c_in, c_out, kernel_size=7, stride=2, padding=3):
        return nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size, stride, padding),
            nn.BatchNorm2d(c_out),
            nn.SiLU(),
        )

    def get_convT_block(self, c_in, c_out, kernel_size=7, stride=2, padding=3, output_padding=1):
        return nn.Sequential(
            nn.ConvTranspose2d(c_in, c_out, kernel_size, stride, padding, output_padding=output_padding),
            nn.BatchNorm2d(c_out),
            nn.SiLU(),
        )

    def forward(self, x, t):
        # x: (b, 1, 28, 28)
        # t: (b,)
        t_embed = self.embed(t)

        z1 = self.conv1(x) # (b, 32, 14, 14)

        z = z1 + self.fc_t1(t_embed)[:, :, None, None]
        z2 = self.conv2(z) # (b, 64, 7, 7)

        z = z2 + self.fc_t2(t_embed)[:, :, None, None]
        z3 = self.conv3(z) # (b, 128, 5, 5)

        z = z3 + self.fc_t3(t_embed)[:, :, None, None]
        z = self.convT3(z) # (b, 64, 7, 7)

        z = torch.cat([z, z2], dim=1) + self.fc_t4(t_embed)[:, :, None, None]
        z = self.convT2(z) # (b, 32, 14, 14)

        z = torch.cat([z, z1], dim=1) + self.fc_t5(t_embed)[:, :, None, None]
        z = self.convT1(z) # (b, 16, 28, 28)

        z = z + self.fc_t6(t_embed)[:, :, None, None]
        z = self.out_conv(z) # (b, 1, 28, 28)

        return z

テスト

In [ ]:
model = UNet().to(device)
x, _ = next(iter(dataloader))
n_steps = 32
t = torch.randint(0, n_steps, (len(x),)) / n_steps
x = x.to(device)
t = t.to(device)
with torch.no_grad():
    z = model(x, t)
print(z.shape)

## Inference

In [ ]:
@torch.inference_mode()
def sample(model, n_samples, n_steps):
    model.eval()
    xt = torch.randn(n_samples, 1, SIZE, SIZE).to(device)
    hist = [xt.cpu()]

    dt = 1 / n_steps
    for t in tqdm(range(n_steps), desc="Sampling"):
        t *= dt
        t = torch.tensor([t] * n_samples).to(device)
        v = model(xt, t)
        xt = xt + dt * v
        hist.append(xt.cpu())
    return xt, hist


def flow_matching(model, n_samples=10, duration=3, n_steps=32):
    _, hist = sample(model, n_samples, n_steps)
    fig, ax = plt.subplots(figsize=(10, 1))
    ims = []
    for x in hist:
        x = torch.clamp(x, 0, 1)
        im = ax.imshow(torchvision.utils.make_grid(x, nrow=10).permute(1, 2, 0))
        ax.axis("off")
        ims.append([im])
    ani = animation.ArtistAnimation(fig, ims, interval=int(duration*1000/len(hist)), repeat=False)
    display(HTML(ani.to_jshtml()))
    plt.close()

テスト

In [ ]:
flow_matching(model)

## Training

In [ ]:
def train(model, dataloader, optimizer, n_epochs=10, n_steps=32):
    model.train()
    prog.start(n_iter=len(dataloader), n_epochs=n_epochs, label="loss")
    for _ in range(n_epochs):
        for x1, _ in dataloader:
            optimizer.zero_grad()
            x1 = x1.to(device)
            x0 = torch.randn(x1.size()).to(device)
            t = torch.randint(0, n_steps, (len(x1),)) / n_steps
            t = t.to(device)
            t_coef = t[:, None, None, None]
            xt = t_coef * x1 + (1 - t_coef) * x0
            target = x1 - x0
            pred = model(xt, t)
            loss = F.mse_loss(pred, target)
            loss.backward()
            optimizer.step()
            prog.update(loss.item())

In [ ]:
model = UNet().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
train(model, dataloader, optimizer, n_epochs=3)

In [ ]:
flow_matching(model, duration=3)